# Spark performance: caching, broadcast joins, Catalyst, AQE, and pushdown

_Apache Spark_

**Reuse work with cache, kill shuffles with broadcast joins, read less with pushdown, and let Catalyst, Tungsten, and AQE do the rest.**

Spark is fast by default because two engines do the heavy lifting under the DataFrame API
       (Application Programming Interface): the Catalyst optimizer and the Tungsten execution engine.
       But you can still write a slow job, and the four biggest levers you control are caching,
       broadcast joins, pushdown, and AQE (Adaptive Query Execution).

       The unifying idea is simple: do less work, and don't do the same work twice.

---

This notebook is a practice scaffold for the **Spark performance: caching, broadcast joins, Catalyst, AQE, and pushdown** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q pyspark
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PySpark

### Step 1 — Start a Spark session with AQE turned on

First we create the `SparkSession`. The four AQE (Adaptive Query Execution) configs let Spark **re-optimize the query at runtime** — after it has seen the real data sizes — instead of committing to a plan up front.

Two of those settings matter most: coalescing merges the many tiny shuffle partitions back into a few sensible ones, and skew-join splits an oversized partition so one straggler task can't hold up the whole stage.

In [ ]:
import time
from pyspark.sql import SparkSession, functions as F
from pyspark.storagelevel import StorageLevel

# Build the session. AQE re-optimizes the query at runtime (on by default in 3.2+).
spark = (SparkSession.builder
         .master("local[*]")
         .appName("spark-performance")
         .config("spark.sql.adaptive.enabled", "true")
         .config("spark.sql.adaptive.coalescePartitions.enabled", "true")  # merge tiny shuffle partitions
         .config("spark.sql.adaptive.skewJoin.enabled", "true")            # split skewed partitions
         .getOrCreate())


### Step 2 — Build an expensive DataFrame and reuse it with cache

We construct a deliberately *expensive* pipeline: generate 4 million rows, filter, derive a column, then `groupBy` (which forces a shuffle). Spark is **lazy**, so this lineage runs nothing yet — it's just a recipe.

The point of `persist` is reuse. Without it, every action would re-run the whole pipeline from scratch. The first action pays the full cost and stores the result; the next two read straight from the cache, so they're nearly free. We `unpersist` at the end to release the memory.

In [ ]:
# Build an "expensive" DataFrame: generate + filter + derive + aggregate.
big = spark.range(0, 4_000_000).select(
    (F.col("id") % 500_000).alias("user"),
    (F.rand(seed=0) * 8).cast("int").alias("region"),
    (F.rand(seed=1) * 200).alias("amount"),
)

expensive = (big
             .filter(F.col("amount") > 5.0)
             .withColumn("logamt", F.log1p("amount"))   # built-in F. function (codegen-friendly)
             .groupBy("region")                         # a shuffle happens here
             .agg(F.count("*").alias("n"), F.sum("logamt").alias("tot")))

# CACHE / PERSIST: avoid recomputing across multiple actions.
# cache() == persist(StorageLevel.MEMORY_AND_DISK). Pick the level explicitly if you like:
expensive = expensive.persist(StorageLevel.MEMORY_AND_DISK)

# Action 1 computes the pipeline AND fills the cache.
t0 = time.perf_counter()
expensive.count()
print("action 1 (computes + caches):", round(time.perf_counter() - t0, 3), "s")

# Action 2 reads the cached result instead of recomputing.
t0 = time.perf_counter()
expensive.agg(F.sum("n")).collect()
print("action 2 (reads cache, fast):", round(time.perf_counter() - t0, 3), "s")

# Action 3 also reads the cache.
t0 = time.perf_counter()
expensive.show(3)
print("action 3 (reads cache, fast):", round(time.perf_counter() - t0, 3), "s")

expensive.unpersist()   # free the memory when done -- don't leave caches pinned


### Step 3 — Broadcast the small side of a join

When you join a huge table to a tiny lookup table, a default join shuffles **both** sides across the network by the key — dragging the big table around is the expensive part.

Wrapping the small table in `F.broadcast` tells Spark to ship a full copy to every executor instead. Each executor then joins its local slice of `big` against the in-memory copy, so the big table never moves. In the plan this becomes a `BroadcastHashJoin` with no shuffle of `big`.

In [ ]:
# BROADCAST JOIN: ship the SMALL side to every executor, no shuffle of the big table.
region_names = spark.createDataFrame(
    [(i, f"region_{i}") for i in range(8)], ["region", "region_name"])   # tiny lookup

# F.broadcast => BroadcastHashJoin (no shuffle of 'big').
joined = big.join(F.broadcast(region_names), "region")
print("rows after broadcast join:", joined.count())


### Step 4 — Write Parquet, then read less with pushdown

Parquet is columnar and stores per-block statistics, which unlocks two kinds of *pushdown*. **Column pushdown**: a `select` reads only the columns you ask for and skips the rest on disk. **Predicate pushdown**: a `filter` lets Spark skip entire row blocks whose statistics prove they can't match.

Finally we call `explain("formatted")` on both DataFrames to read the physical plan — look for `PushedFilters` on the pushdown read and `BroadcastHashJoin` on the join. Then we stop the session.

In [ ]:
# PARQUET + PUSHDOWN: read only the columns and rows you need.
big.write.mode("overwrite").parquet("/tmp/events.parquet")

pushed = (spark.read.parquet("/tmp/events.parquet")
          .select("user", "amount")        # COLUMN pushdown: other columns are never read
          .filter(F.col("amount") > 150))  # PREDICATE pushdown: row blocks that can't match are skipped
print("pushed-down rows:", pushed.count())

# EXPLAIN: read the physical plan.
# Look for BroadcastHashJoin, PushedFilters, WholeStageCodegen.
pushed.explain("formatted")
joined.explain("formatted")

spark.stop()


## Visualize the data & results

_When a DataFrame is reused across several actions, how much does cache() save versus recomputing the whole pipeline each time?_

### Step 1 — Build a dataset and define the expensive pipeline

We can't time Spark's cache cheaply in a quick demo, so we mirror the idea with pandas. We create 4 million rows, then define `expensive_pipeline` — a filter, a derived column, and a `groupby` aggregation — which stands in for the lineage Spark would **recompute on every action** unless you cache it.

`timeit` is a tiny helper that runs a function and returns how many milliseconds it took.

In [ ]:
import numpy as np
import pandas as pd
import time

rng = np.random.RandomState(0)
N = 4_000_000
df = pd.DataFrame({
    "user":   rng.randint(0, 500_000, size=N),
    "amount": rng.exponential(40.0, size=N).round(2),
    "region": rng.randint(0, 8, size=N),
})

# An "expensive" lineage Spark would RECOMPUTE on every action unless cached.
def expensive_pipeline(d):
    out = d[d["amount"] > 5.0].copy()
    out["logamt"] = np.log1p(out["amount"])
    return out.groupby("region", as_index=False).agg(
        n=("amount", "size"), tot=("logamt", "sum"))

def timeit(fn, *a):
    t0 = time.perf_counter()
    fn(*a)
    return (time.perf_counter() - t0) * 1000.0   # ms


### Step 2 — Time the work without a cache, then with one

Without caching, three actions each re-run the *full* pipeline from the source — so all three pay roughly the same cost. With caching, you pay the pipeline **once** to materialize the small result, then every later read of that result is almost free.

The printed speedup compares a single reused read with and without the cache — it should be enormous, because the cached read does essentially no work.

In [ ]:
# WITHOUT cache: three actions each re-run the full pipeline from source.
no_cache = [round(timeit(expensive_pipeline, df), 2) for _ in range(3)]

# WITH cache: pay the pipeline ONCE to materialize, then read the small result near-free.
cached = expensive_pipeline(df)                      # read 1 fills the cache
first  = round(timeit(expensive_pipeline, df), 2)    # cost to compute + cache
read   = lambda: cached["tot"].sum()                 # trivial action on the cached result
second = round(timeit(read), 3)
third  = round(timeit(read), 3)
with_cache = [first, second, third]

print("no_cache  (ms):", no_cache)    # ~ [110, 100, 98]
print("with_cache(ms):", with_cache)  # ~ [97, 0.07, 0.02]
print("speedup on a reused read:", round(no_cache[1] / max(with_cache[1], 1e-9), 0), "x")


## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** You define df from a Parquet read plus a filter and two derived columns, then call df.count(), write df to disk, and finally df.show(). The job is slow and the Spark UI shows the Parquet scan running three times. What is happening and what is the one-line fix?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall that Spark DataFrames are lazy: defining df runs nothing. — _The transform chain is only a recipe until an action forces it._
- Count three actions: count(), the write, and show(). — _Each action independently re-executes the whole chain from the source, so the expensive Parquet scan runs once per action._
- Insert df = df.cache() before the first action. — _The first action materializes and stores the result; the next two read from memory instead of re-scanning._

**Answer:** Because Spark is lazy, each of the three actions recomputes df from the Parquet source &mdash; hence three scans. Add df = df.cache() (or persist(...)) before the first action: the count() computes it once and caches it, and the write and show() reuse the cached result. Roughly a 3x cut on the repeated work. Call df.unpersist() when finished to release the memory.

</details>

**Problem 2.** You join a 2-billion-row events table to a 5,000-row country_lookup table on country_code. The stage shuffles hundreds of GB and is by far the slowest in the job. How do you fix it, and why does it work?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Notice one side is tiny (5,000 rows) and the other is enormous. — _A default join shuffles BOTH tables by the key, dragging the 2B-row table across the network &mdash; the expensive part._
- Broadcast the small side: events.join(F.broadcast(country_lookup), "country_code"). — _Spark ships a full copy of the 5,000-row table to every executor; each executor joins its local slice of events against it._
- Confirm with .explain() that the plan now shows BroadcastHashJoin, not SortMergeJoin. — _The broadcast plan has no exchange/shuffle on the big table._

**Answer:** Wrap the small table in F.broadcast(country_lookup). Spark sends a full copy of the 5,000-row lookup to every executor, so each one joins its local partition of events against the in-memory copy &mdash; no shuffle of the 2-billion-row table. The shuffle was the whole cost, so this can turn a long stage into a quick one. (With AQE on, Spark may even do this automatically once it sees the small side's true size.)

</details>

**Problem 3.** A colleague speeds up a string-cleaning step by writing a Python UDF (User-Defined Function) that lowercases and strips punctuation per row. After the change the stage got slower, and explain() no longer shows whole-stage codegen around it. Why, and what should they do instead?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall that Tungsten compiles a chain of built-in operators into one tight generated-code loop (whole-stage codegen). — _That codegen plus off-heap binary rows is what makes DataFrame operations fast._
- Note a Python UDF is a black box the engine cannot see into. — _Spark must serialize each row to Python, run the function row-by-row, and serialize back &mdash; breaking codegen and adding per-row overhead._
- Replace the UDF with built-in F. functions: F.lower(F.regexp_replace(col, "[^a-z0-9 ]", "")). — _Built-ins stay inside the engine, so codegen and off-heap execution are preserved._

**Answer:** A Python UDF is opaque to Catalyst and Tungsten, so the engine cannot fold it into whole-stage codegen. Every row is serialized out to Python, processed one at a time, and serialized back &mdash; which both breaks the generated-code loop and adds heavy per-row overhead, often 10x slower. The fix is to express the same logic with built-in F. functions (here F.lower and F.regexp_replace), which run natively inside the engine and keep codegen intact.

</details>